In [ ]:
from pathlib import Path
import os
import sys

import mne
import matplotlib.pyplot as plt
import numpy as np

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

In [ ]:
from modules.plotting import plot_tep_components_topomap, plot_tep
from modules.processing import TEPAnalysis
from modules.preprocessing import interpolate_channels
from config import eeg_motor_rois

In [ ]:
epochs = mne.read_epochs("data/processed/epochs-tep.fif", preload=True)

In [ ]:
epochs.drop_channels(['EOG', 'EMG L', 'EMG R'])

In [ ]:
# Informe os canais EEG removidos no pre-processamento desta gravacao.
# Exemplo: bad_ch = ["TP10", "Iz", "T7", "FC1", "TP9", "T8"]
bad_ch = []

# Reconstrua os canais antes de calcular a densidade de corrente (CSD).
if bad_ch:
    epochs = interpolate_channels(epochs, bad_ch, montage="standard_1020")
    epochs.set_eeg_reference("average", projection=False)

In [ ]:
epochs_csd = mne.preprocessing.compute_current_source_density(epochs.copy())

In [1]:
plot_tep(epochs_csd['tms_pulse_task_right'].pick(eeg_motor_rois['Premotor_L']))
plot_tep(epochs_csd['tms_pulse_task_left'].pick(eeg_motor_rois['Premotor_L']))
plot_tep(epochs_csd['tms_pulse_task_bilateral'].pick(eeg_motor_rois['Premotor_L']))

NameError: name 'plot_tep' is not defined

In [ ]:
tep = TEPAnalysis(epochs_csd)

In [ ]:
df = tep.get_significant_channels_snr(baseline = (-0.250, -0.01), conditions = 'tms_pulse_task_right', threshold = 5)
channels = df[df['is_significant'] == True]['channel'].to_list()